<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/193-Langchain-RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Necessary Libraries
First, we need to install the required libraries: `langchain_community` for web loading and text splitting, `chromadb` for the vector store, and `langchain_openai` for embeddings and the language model. If you prefer to use open-source models, you can substitute `langchain_openai` with libraries like `langchain_huggingface` and `sentence-transformers`.



In [1]:
!pip install -qqq langchain_community chromadb langchain-openai langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 105.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.

### 2. Set Up API Key (if using OpenAI)
If you plan to use OpenAI's models, you'll need to set up your API key. In Colab, you can store it securely in the secrets manager and access it as shown below. If you're using a different LLM provider or a local model, you'll configure that instead.

In [2]:
# Used to securely store your API key
from google.colab import userdata

import os

# Set your OpenAI API key. If running in Colab, store it securely in secrets.
# If running locally or programmatically, you will need to set it as an environment variable.
# For example, before running this cell, you might do:
# os.environ['OPENAI_API_KEY'] = 'your_openai_api_key_here'

if 'OPENAI_API_KEY' not in os.environ:
    # This line attempts to get the key from Colab secrets, which might time out
    # if not running interactively in the Colab UI.
    try:
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception as e:
        print(f"Warning: Could not retrieve API key from Colab secrets: {e}")
        print("Please ensure your OPENAI_API_KEY is set as an environment variable or in Colab secrets.")

if 'OPENAI_API_KEY' in os.environ:
    print("OPENAI_API_KEY is set.")
else:
    print("OPENAI_API_KEY is NOT set. Please set it to proceed.")

OPENAI_API_KEY is set.


### 3. Load the Document from the URL
We'll use `WebBaseLoader` from `langchain_community` to fetch the content from the specified URL.

In [3]:
from langchain_community.document_loaders import WebBaseLoader

# Define the URL to load
url = "http://www.paulgraham.com/fr.html"

# Initialize the WebBaseLoader with the URL
loader = WebBaseLoader(url)

# Load the documents
docs = loader.load()

print(f"Loaded {len(docs)} document(s).")
print("First 500 characters of the document:\n")
print(docs[0].page_content[:500])


/tmp/ipykernel_826/813983692.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


Loaded 1 document(s).
First 500 characters of the document:

How to Raise Money



Want to start a startup?  Get funded by
Y Combinator.




September 2013Most startups that raise money do it more than once.  A typical
trajectory might be (1) to get started with a few tens of thousands
from something like Y Combinator or individual angels, then 
(2) raise a few hundred thousand to a few million to build the company,
and then (3) once the company is clearly succeeding, raise one or
more later rounds to accelerate growth.Reality can be messier.  Some compan


In [4]:
len(docs)

1

### 4. Split the Document into Chunks
To effectively retrieve relevant information, we need to split the large document into smaller, manageable chunks. We'll use `RecursiveCharacterTextSplitter` for this.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Maximum size of each chunk
    chunk_overlap=200,    # Overlap between chunks to maintain context
    add_start_index=True, # Add a metadata field for the start index of the chunk
)

# Split the documents
all_splits = text_splitter.split_documents(docs)

print(f"Split into {len(all_splits)} chunks.")
print("First chunk:\n")
print(all_splits[0].page_content[:500])


Split into 80 chunks.
First chunk:

How to Raise Money



Want to start a startup?  Get funded by
Y Combinator.


In [6]:
print(all_splits[79].page_content[:500])

[27]
The actual sentence in the King James Bible is "Pride goeth
before destruction, and an haughty spirit before a fall."Thanks to Slava Akhmechet, Sam Altman, Nate Blecharczyk,
Adora Cheung, Bill Clerico, John Collison, Patrick Collison, Parker
Conrad, Ron Conway, Travis Deyle, Jason Freedman, Joe Gebbia, Mattan
Griffel, Kevin Hale, Jacob Heller, Ian Hogarth, Justin Kan, Professor
Moriarty, Nikhil Nirmel, David Petersen, Geoff Ralston, Joshua
Reeves, Yuri Sagalov, Emmett Shear, Rajat Suri, Gar


### 5. Create Embeddings and a Vector Store
Next, we'll generate vector embeddings for each document chunk using OpenAIEmbeddings and store them in a Chroma vector database. This allows us to perform semantic search on the document content.

In [7]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Initialize the embeddings model
embeddings = OpenAIEmbeddings()

# Create a Chroma vector store from the document chunks and embeddings
vectorstore = Chroma.from_documents(documents=all_splits, embedding=embeddings)

print("Vector store created successfully.")


Vector store created successfully.


In [8]:
print(embeddings.model)


text-embedding-ada-002


### 6. Set Up the RAG Chain
Now, we'll combine the retriever (which fetches relevant document chunks from the vector store) with a language model (OpenAI's ChatOpenAI) to build our RAG chain. This chain will retrieve relevant context and then use it to answer user questions.

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)

# Convert the vector store into a retriever
retriever = vectorstore.as_retriever()

In [10]:


# Define the RAG prompt template directly (bypassing hub.pull)
# This is a common RAG prompt pattern
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer the user's question based on the below context:\n\n{context}"),
    ("human", "{question}"),
])

# Define format_docs BEFORE it is used in rag_chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Construct the RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


In [12]:
rag_chain.invoke("고양이 다리의 갯수는?")

'고양이는 일반적으로 4개의 다리를 가지고 있습니다.'

### 7. Ask a Question
Finally, let's test our RAG system by asking a question related to the content of the Paul Graham essay.

In [13]:
question = "What are some key insights about founding a startup mentioned in the document?"
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)


Question: What are some key insights about founding a startup mentioned in the document?

Answer:
Some key insights about founding a startup mentioned in the document include:

1. **Fundraising Challenges**: Fundraising is inherently difficult, as it involves convincing investors to part with large sums of money. Founders should be aware that the process can be fraught with delays and uncertainties.

2. **Investor Behavior**: Investors can be unpredictable, and even established firms may back out of deals. Founders should be cautious and avoid investors who do not take a leading role in the investment process.

3. **Timeliness is Crucial**: Once an investor commits, it’s important to understand the timeline for receiving funds and to actively manage that process to ensure the money is secured.

4. **Inexperienced Investors**: Newer investors are more likely to experience buyer's remorse, so founders should be particularly attentive when dealing with them.

5. **External Constraints**: 

In [14]:
question = "저자의 이름은?"
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)

Question: 저자의 이름은?

Answer:
저자의 이름은 명시되어 있지 않습니다. 그러나 문맥에서 여러 사람들의 이름이 언급되고 있으며, 이들은 저자의 글을 읽어본 사람들입니다.


In [15]:
question = "문자의 핵심 메세지를 100자로 정리해 주세요."
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)

Question: 문자의 핵심 메세지를 100자로 정리해 주세요.

Answer:
투자자 미팅은 밀집하게 잡되, 피치를 발전시킬 시간도 필요하다. 다양한 자금 조달 계획을 준비하고, 투자자의 관심은 신뢰하지 말고 확실한 약속을 기다려야 한다.


In [16]:
question = "고양이 다리의 갯수는?"
response = rag_chain.invoke(question)

print("Question:", question)
print("\nAnswer:")
print(response)

Question: 고양이 다리의 갯수는?

Answer:
고양이의 다리의 갯수는 4개입니다.


In [17]:
from langchain_core.documents import Document

# New text to add to the Chroma DB
new_text = "고양이 다리는 3개 입니다."

# Create a new Document object
new_doc = Document(page_content=new_text)

# Add the new document to the vectorstore
vectorstore.add_documents([new_doc])

print(f"Successfully added new document: '{new_text}' to the vectorstore.")
print(f"Current number of documents in vectorstore: {vectorstore._collection.count()}")

Successfully added new document: '고양이 다리는 3개 입니다.' to the vectorstore.
Current number of documents in vectorstore: 81


In [18]:


# You can now test by asking a question related to this new information
question_new = "고양이 다리는 몇 개라고 언급되어 있나요?"
response_new = rag_chain.invoke(question_new)
print("\nQuestion (new):", question_new)
print("\nAnswer (new):")
print(response_new)


Question (new): 고양이 다리는 몇 개라고 언급되어 있나요?

Answer (new):
고양이 다리는 3개라고 언급되어 있습니다.
